<a href="https://colab.research.google.com/github/Mac-Tapia/MADRLCitytleranflexresdr/blob/codex/fix-madrl-traceability-docs/examples_madrl_v3/madrl_citylearn_v3_cli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CLI CityLearn v3 MADRL

Este notebook sigue la estructura del ejemplo `cli.ipynb` original de CityLearn, pero esta adaptado al proyecto **CityLearn v3 MADRL**: 17 edificios + EV, tres ejes `E1/E2/E3`, recompensa `CityLearnV3MADRLRewardFunction`, backends oficiales HAPPO, MASAC, MATD3 y MAAC, y artefactos reproducibles de entrenamiento.

En CityLearn v2 el CLI principal se invoca con `python -m citylearn`. En este proyecto se mantiene ese CLI base para datasets, variables y simulaciones compatibles, y se agregan scripts de entrenamiento MADRL en `scripts/train_citylearn_v3_*.py` mas un lanzador PowerShell para la corrida oficial secuencial.

## Install or Open the Project

En una maquina local del proyecto no es necesario reinstalar. En Google Colab se debe clonar la rama del proyecto e instalarla en modo editable. La celda queda apagada por defecto para no modificar el entorno accidentalmente.

In [ ]:
# Controla si se ejecuta la instalacion en Colab.
RUN_COLAB_INSTALL = False

# Importa utilidades estandar usadas por las celdas CLI.
import os

# Importa rutas portables para Windows, Linux y Colab.
from pathlib import Path

# Ejecuta comandos externos cuando se habiliten las celdas.
import subprocess

# Expone el interprete Python activo.
import sys

# Instala el proyecto solo si el usuario activa la bandera.
if RUN_COLAB_INSTALL:
    # Clona la rama publica del proyecto CityLearn v3 MADRL.
    subprocess.run(['git', 'clone', '--branch', 'citylearn-v3-madrl', 'https://github.com/Mac-Tapia/CityLearn.git'], check=True)
    # Cambia el directorio de trabajo al repositorio clonado.
    os.chdir('CityLearn')
    # Instala CityLearn en modo editable para usar los scripts locales.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
# Maneja el caso normal en que no se instala nada.
else:
    # Informa que la celda no hizo cambios en el entorno.
    print('RUN_COLAB_INSTALL=False; usando el entorno actual.')

## Project Paths

La celda detecta si el notebook se ejecuta desde la raiz del repositorio o desde `examples/`. Las rutas resultantes se usan para construir comandos CLI reproducibles.

In [ ]:
# Obtiene el directorio actual donde se ejecuto el notebook.
current_path = Path.cwd().resolve()

# Detecta la raiz CityLearn cuando el notebook corre desde examples.
if current_path.name == 'examples' and (current_path.parent / 'scripts').is_dir():
    # Usa la carpeta padre como raiz CityLearn.
    project_root = current_path.parent
# Detecta la raiz CityLearn cuando el notebook corre desde CityLearn/.
elif (current_path / 'scripts').is_dir() and (current_path / 'citylearn').is_dir():
    # Usa el directorio actual como raiz CityLearn.
    project_root = current_path
# Detecta la raiz del superproyecto que contiene el subdirectorio CityLearn/.
elif (current_path / 'CityLearn' / 'scripts').is_dir():
    # Usa el subdirectorio CityLearn como raiz ejecutable.
    project_root = current_path / 'CityLearn'
# Mantiene el directorio actual como ultimo recurso.
# Maneja el caso seguro en que no se lanza entrenamiento.
else:
    # Usa el directorio actual y deja que las validaciones de rutas fallen visiblemente.
    project_root = current_path

# Define la carpeta de scripts CityLearn v3 MADRL.
scripts_dir = project_root / 'scripts'

# Define la carpeta de salidas para pruebas CLI cortas.
cli_output = project_root / 'outputs' / 'citylearn_v3_madrl_cli_demo'

# Define los algoritmos oficiales integrados.
algorithms = ['happo', 'masac', 'matd3', 'maac']

# Define los tres ejes del proyecto.
scenarios = ['E1', 'E2', 'E3']

# Muestra las rutas clave para diagnostico.
print('project_root =', project_root)

# Muestra la carpeta de scripts.
print('scripts_dir =', scripts_dir)

# Muestra la carpeta de salidas.
print('cli_output =', cli_output)

## CityLearn Base CLI Documentation

El CLI base de CityLearn v2 sigue disponible. Sirve para listar datasets, variables por defecto y ejecutar agentes compatibles con `citylearn.agents` o `stable-baselines3`.

In [ ]:
# Construye el comando de ayuda del CLI base.
citylearn_help_command = [sys.executable, '-m', 'citylearn', '-h']

# Muestra el comando que se ejecutara.
print('Running:', ' '.join(citylearn_help_command))

# Ejecuta la ayuda del CLI base de CityLearn.
subprocess.run(citylearn_help_command, check=False, cwd=project_root)

## Running a Simulation using the Original CLI

La suborden `simulate` es parte de CityLearn v2 y se conserva para agentes base, RBC, agentes internos y evaluaciones compatibles. Para los cuatro MADRL del proyecto se usan los scripts `train_citylearn_v3_*.py`, porque esos backends requieren adaptadores Dec-POMDP/CTDE y artefactos adicionales.

In [ ]:
# Construye el comando de ayuda de la suborden simulate.
simulate_help_command = [sys.executable, '-m', 'citylearn', 'simulate', '-h']

# Muestra el comando que se ejecutara.
print('Running:', ' '.join(simulate_help_command))

# Ejecuta la ayuda de simulate.
subprocess.run(simulate_help_command, check=False, cwd=project_root)

## CityLearn v3 MADRL Training CLIs

Cada algoritmo tiene su propio script porque cada backend oficial usa una naturaleza de entrenamiento distinta:

- `train_citylearn_v3_happo.py`: HAPPO on-policy con critico centralizado HARL.
- `train_citylearn_v3_masac.py`: MASAC con estado global CTDE y politica local discretizada.
- `train_citylearn_v3_matd3.py`: MATD3 off-policy continuo con criticos centralizados.
- `train_citylearn_v3_maac.py`: MAAC con critico de atencion multiagente.

Las ayudas `-h` muestran los parametros disponibles sin iniciar entrenamiento.

In [ ]:
# Controla si se muestran las ayudas de los cuatro scripts MADRL.
SHOW_MADRL_HELP = True

# Recorre los cuatro algoritmos oficiales del proyecto.
for algorithm in algorithms:
    # Construye la ruta del script de entrenamiento.
    script_path = scripts_dir / f'train_citylearn_v3_{algorithm}.py'
    # Construye el comando de ayuda del script.
    command = [sys.executable, '-B', str(script_path), '-h']
    # Muestra el comando de ayuda para trazabilidad.
    print('\n' + algorithm.upper())
    # Muestra el comando exacto que se ejecutaria.
    print(' '.join(map(str, command)))
    # Ejecuta la ayuda si la bandera esta activa.
    if SHOW_MADRL_HELP:
        # Ejecuta la ayuda sin iniciar entrenamiento.
        subprocess.run(command, check=False, cwd=project_root)

## Official Full Training Command

El entrenamiento oficial usa 17 edificios + EV, dataset `citylearn_iquitos_2023_2025`, escenarios `E1/E2/E3`, 8760 pasos por episodio y 5 episodios. En Windows el flujo oficial se lanza con PowerShell y ejecuta secuencialmente los 12 trabajos `E1/E2/E3 x HAPPO/MASAC/MATD3/MAAC`.

La celda no ejecuta el entrenamiento largo salvo que `RUN_OFFICIAL_TRAINING=True`.

In [ ]:
# Controla si se lanza el entrenamiento oficial completo.
RUN_OFFICIAL_TRAINING = False

# Define la carpeta oficial de salidas v4 (entrenamiento activo 2026-06-08).
official_output = project_root / 'outputs' / 'citylearn_v3_madrl_oficial_v4'

# Construye el comando PowerShell oficial.
official_command = [
    # Ejecuta PowerShell sin perfiles de usuario.
    'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass',
    # Llama al lanzador oficial del proyecto.
    '-File', str(scripts_dir / 'launch_citylearn_v3_official_training.ps1'),
    # Ejecuta los tres ejes E1, E2 y E3.
    '-Scenario', 'ALL',
    # Usa la semilla base del proyecto.
    '-Seed', '0',
    # Usa el horizonte anual completo.
    '-EpisodeTimeSteps', '8760',
    # Usa 50 episodios por corrida (reanudable con --skip-completed).
    '-Episodes', '17',
    # Escribe resultados en la carpeta oficial v4.
    '-OutputRoot', str(official_output),
    # Perfil GPU para RTX 4060 Laptop 8 GB (PyTorch 2.8.0+cu126).
    '-GpuProfile', 'local4060',
    # Usa hilos Torch para los backends PyTorch.
    '-TorchThreads', '12',
    # Activa CUDA cuando PyTorch detecta GPU compatible.
    '-Cuda',
    # Muestra progreso en la ventana actual.
    '-LiveOutput',
    # Salta corridas ya completadas (results.json existente).
    '-SkipCompleted',
]

# Muestra el comando oficial completo.
print(' '.join(map(str, official_command)))

# Ejecuta el entrenamiento solo si el usuario activa la bandera.
if RUN_OFFICIAL_TRAINING:
    # Lanza el proceso oficial en la terminal activa.
    subprocess.run(official_command, check=True, cwd=project_root)
# Maneja el caso seguro en que no se lanza entrenamiento.
else:
    # Mantiene la celda segura por defecto.
    print('RUN_OFFICIAL_TRAINING=False; no se lanzo el entrenamiento largo.')

## Linux or Google Colab GPU Command

Colab no usa PowerShell. Para Colab A100/T4/V100 se ejecutan directamente los scripts Python por escenario y algoritmo. Esta celda imprime comandos equivalentes con los hiperparametros oficiales actuales del proyecto.

In [ ]:
# Define episodios de entrenamiento para comandos Linux/Colab.
episodes = 50

# Define pasos por episodio oficiales.
episode_time_steps = 8760

# Calcula pasos totales de entorno.
num_env_steps = episodes * episode_time_steps

# Construye un comando Python oficial por algoritmo y escenario.
# Hiperparametros alineados al perfil local4060 activo (RTX 4060 Laptop 8 GB, PyTorch 2.8.0+cu126).
def build_madrl_command(algorithm: str, scenario: str) -> list[str]:
    # Construye la ruta del script del algoritmo.
    script_path = scripts_dir / f'train_citylearn_v3_{algorithm}.py'
    # Define argumentos comunes a los cuatro MADRL.
    command = [sys.executable, '-B', str(script_path), '--scenario', scenario, '--seed', '0', '--episode-time-steps', str(episode_time_steps), '--episodes', str(episodes), '--output-dir', str(cli_output / algorithm), '--cuda']
    # Agrega hiperparametros oficiales de HAPPO.
    if algorithm == 'happo':
        # HAPPO on-policy, red 512 unidades, n_rollout_threads=1.
        command += ['--num-env-steps', str(num_env_steps), '--hidden-size', '512', '--torch-threads', '8']
    # Agrega hiperparametros oficiales de MASAC.
    elif algorithm == 'masac':
        # MASAC: buffer_size=4, critic_batch_size=4, rnn_hidden_dim=128 (perfil local4060).
        command += ['--action-bins', '3', '--buffer-size', '4', '--critic-batch-size', '4',
                    '--rnn-hidden-dim', '128', '--qmix-hidden-dim', '64', '--hyper-hidden-dim', '96']
    # Agrega hiperparametros oficiales de MATD3.
    elif algorithm == 'matd3':
        # MATD3 off-policy: batch=1024, buffer=8192, hidden=384, train_interval=50.
        command += ['--num-env-steps', str(num_env_steps), '--batch-size', '1024',
                    '--buffer-size', '8192', '--hidden-size', '384',
                    '--train-interval', '50', '--num-random-episodes', '1']
    # Agrega hiperparametros oficiales de MAAC.
    elif algorithm == 'maac':
        # MAAC: batch=256, buffer_length=512, hidden=256, atencion 4 cabezas.
        command += ['--action-bins', '3', '--batch-size', '256', '--buffer-length', '512',
                    '--steps-per-update', '100', '--num-updates', '4', '--hidden-size', '256',
                    '--attend-heads', '4', '--pi-lr', '0.0003', '--q-lr', '0.001',
                    '--tau', '0.005', '--gamma', '0.99']
    # Devuelve el comando listo para ejecutar.
    return command

# Imprime la matriz completa de comandos sin ejecutarla.
for scenario in scenarios:
    # Imprime encabezado del eje actual.
    print('\nScenario', scenario)
    # Recorre los cuatro MADRL.
    for algorithm in algorithms:
        # Imprime el comando exacto para el par escenario-algoritmo.
        print(' '.join(map(str, build_madrl_command(algorithm, scenario))))

## Monitor Visual CLI

El monitor del proyecto lee `official_full_status.json`, logs, `live_progress.json`, `timeseries.csv`, `trace.csv`, checkpoints y estado GPU. Muestra el avance por `E1/E2/E3`, MADRL activo, recompensa instantanea, retorno acumulado, funcion reward, perfil MADRL, pesos activos, costo, CO2 y carga neta.

In [ ]:
# Construye el comando del monitor visual PowerShell.
monitor_command = [
    # Ejecuta PowerShell sin perfiles de usuario.
    'powershell.exe', '-NoProfile', '-ExecutionPolicy', 'Bypass',
    # Llama al monitor oficial del proyecto.
    '-File', str(scripts_dir / 'monitor_citylearn_v3_official_training.ps1'),
    # Indica la carpeta de salidas oficiales.
    '-OutputRoot', str(official_output),
    # Actualiza cada cinco segundos.
    '-IntervalSeconds', '5',
    # Muestra las ultimas lineas de logs.
    '-LogTail', '20',
]

# Muestra el comando del monitor.
print(' '.join(map(str, monitor_command)))

## Outputs and Validation

Cada corrida MADRL debe dejar una carpeta por algoritmo y escenario con `results.json`, `timeseries.csv`, `trace.csv`, `checkpoint_manifest.json`, `checkpoints/`, `figures/` y tablas por eje. La validacion de perfiles reward confirma que los pesos de entrenamiento pertenecen a CityLearn v3 y no a la recompensa MARL base.

In [ ]:
# Construye el comando de validacion de perfiles reward.
reward_validation_command = [sys.executable, '-B', str(scripts_dir / 'validate_citylearn_v3_reward_profiles.py'), '--output', str(project_root / 'outputs' / 'citylearn_v3_reward_profile_validation.json')]

# Construye el comando de validacion de objetivos para un escenario corto.
objective_validation_command = [sys.executable, '-B', str(scripts_dir / 'validate_citylearn_v3_objectives.py'), '--scenario', 'E1', '--seed', '0', '--episode-time-steps', '24', '--include-citylearn-v2-test-agents', '--output-dir', str(project_root / 'outputs' / 'citylearn_v3_cli_objective_validation')]

# Muestra el comando de validacion de reward.
print('Reward validation:')

# Imprime el comando exacto de reward.
print(' '.join(map(str, reward_validation_command)))

# Muestra el comando de validacion de objetivos.
print('\nObjective validation:')

# Imprime el comando exacto de objetivos.
print(' '.join(map(str, objective_validation_command)))

# Lista los artefactos esperados por corrida.
expected_artifacts = ['results.json', 'training_summary.json', 'timeseries.csv', 'trace.csv', 'checkpoint_manifest.json', 'checkpoints/', 'figures/']

# Muestra los artefactos esperados.
print('\nExpected artifacts:', expected_artifacts)